# 🫀 3D CNN Heart Condition Classifier — Kaggle Training

Train ResNet3D-18 to classify facial videos into 3 heart conditions:
- **normal** (60-80 BPM)
- **abnormal** (arrhythmia, 40-120 BPM)
- **infarction** (MI, 90-130 BPM)

**Instructions:**
1. Upload `video_clips.zip` to Kaggle as a Dataset
2. Enable GPU: Settings → Accelerator → GPU T4×2
3. Run all cells

In [ ]:
import os, cv2, torch, numpy as np
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision.models.video import r3d_18, R3D_18_Weights
from sklearn.model_selection import train_test_split
from pathlib import Path
from collections import Counter
import time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

## 1. Configuration

In [ ]:
# ══════ CONFIG ══════
DATA_DIR = '/kaggle/input/heart-face-video-clips/video_clips'  # Change to your dataset path
OUTPUT_DIR = '/kaggle/working'
NUM_CLASSES = 3
NUM_FRAMES = 16
IMAGE_SIZE = 224
BATCH_SIZE = 8
FREEZE_EPOCHS = 5
TOTAL_EPOCHS = 40
LR_PHASE1 = 1e-3
LR_PHASE2 = 5e-5
WEIGHT_DECAY = 1e-5
PATIENCE = 10
LABELS = ['normal', 'abnormal', 'infarction']
LABEL_MAP = {name: i for i, name in enumerate(LABELS)}

## 2. Dataset

In [ ]:
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
VIDEO_EXTS = {'.mp4', '.avi', '.mov', '.mkv', '.webm'}

class VideoClipDataset(Dataset):
    def __init__(self, clip_paths, labels, num_frames=16, image_size=224, is_train=False):
        self.clip_paths = clip_paths
        self.labels = labels
        self.num_frames = num_frames
        self.image_size = image_size
        self.is_train = is_train
        self.mean = np.array([0.485, 0.456, 0.406])
        self.std = np.array([0.229, 0.224, 0.225])

    def __len__(self): return len(self.clip_paths)

    def __getitem__(self, idx):
        path = Path(self.clip_paths[idx])
        label = self.labels[idx]
        if path.is_dir():
            frames = self._load_folder(path)
        elif path.suffix.lower() in VIDEO_EXTS:
            frames = self._load_video(path)
        else:
            frames = self._pseudo_frames(path)
        processed = []
        for f in frames:
            f = cv2.resize(f, (self.image_size, self.image_size))
            f = cv2.cvtColor(f, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
            if self.is_train:
                if np.random.random() > 0.5:
                    f = np.fliplr(f).copy()
                f = f + np.random.randn(*f.shape).astype(np.float32) * 0.01
            f = (f - self.mean) / self.std
            processed.append(torch.from_numpy(f.astype(np.float32)).permute(2, 0, 1))
        video = torch.stack(processed, dim=1)  # (C, T, H, W)
        return video, label

    def _load_folder(self, folder):
        files = sorted([f for f in folder.iterdir() if f.suffix.lower() in IMAGE_EXTS])
        if not files: return [np.zeros((self.image_size, self.image_size, 3), dtype=np.uint8)] * self.num_frames
        indices = np.linspace(0, len(files)-1, self.num_frames, dtype=int)
        frames = []
        for i in indices:
            img = cv2.imread(str(files[i]))
            frames.append(img if img is not None else np.zeros((self.image_size, self.image_size, 3), dtype=np.uint8))
        return frames

    def _load_video(self, path):
        cap = cv2.VideoCapture(str(path))
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if total <= 0: cap.release(); return [np.zeros((self.image_size, self.image_size, 3), dtype=np.uint8)] * self.num_frames
        indices = np.linspace(0, total-1, self.num_frames, dtype=int)
        frames = []
        for fi in indices:
            cap.set(cv2.CAP_PROP_POS_FRAMES, fi)
            ret, frame = cap.read()
            frames.append(frame if ret else np.zeros((self.image_size, self.image_size, 3), dtype=np.uint8))
        cap.release()
        while len(frames) < self.num_frames: frames.append(frames[-1])
        return frames[:self.num_frames]

    def _pseudo_frames(self, img_path):
        img = cv2.imread(str(img_path))
        if img is None: return [np.zeros((self.image_size, self.image_size, 3), dtype=np.uint8)] * self.num_frames
        img = cv2.resize(img, (self.image_size, self.image_size))
        h, w = img.shape[:2]; center = (w//2, h//2)
        frames = []
        for i in range(self.num_frames):
            angle = np.sin(i * 0.5) * 1.5 if self.is_train else (i - self.num_frames//2) * 0.3
            M = cv2.getRotationMatrix2D(center, angle, 1.0 + np.sin(i*0.3)*0.02)
            frames.append(cv2.warpAffine(img, M, (w, h), borderMode=cv2.BORDER_REFLECT))
        return frames

In [ ]:
# Scan dataset
def scan_dataset(data_dir):
    paths, labels = [], []
    for name, lid in LABEL_MAP.items():
        label_dir = Path(data_dir) / name
        if not label_dir.exists(): print(f'⚠️ Missing: {label_dir}'); continue
        for item in sorted(label_dir.iterdir()):
            if item.is_dir() or item.suffix.lower() in IMAGE_EXTS | VIDEO_EXTS:
                paths.append(str(item)); labels.append(lid)
    print(f'Total: {len(paths)} clips')
    for name, lid in LABEL_MAP.items():
        print(f'  {name}: {labels.count(lid)}')
    return paths, labels

paths, labels = scan_dataset(DATA_DIR)

# Split
train_p, temp_p, train_l, temp_l = train_test_split(paths, labels, test_size=0.3, stratify=labels, random_state=42)
val_p, test_p, val_l, test_l = train_test_split(temp_p, temp_l, test_size=0.5, stratify=temp_l, random_state=42)

train_ds = VideoClipDataset(train_p, train_l, NUM_FRAMES, IMAGE_SIZE, is_train=True)
val_ds = VideoClipDataset(val_p, val_l, NUM_FRAMES, IMAGE_SIZE)
test_ds = VideoClipDataset(test_p, test_l, NUM_FRAMES, IMAGE_SIZE)

train_loader = DataLoader(train_ds, BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, BATCH_SIZE, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_ds, BATCH_SIZE, num_workers=2, pin_memory=True)

print(f'\nSplit: train={len(train_ds)}, val={len(val_ds)}, test={len(test_ds)}')

# Class weights
counts = Counter(train_l)
total = len(train_l)
class_weights = torch.tensor([total / (NUM_CLASSES * counts[i]) for i in range(NUM_CLASSES)]).to(device)
print(f'Class weights: {class_weights}')

## 3. Model

In [ ]:
class HeartClassifier3D(nn.Module):
    def __init__(self, num_classes=3, pretrained=True, dropout=0.3):
        super().__init__()
        self.num_classes = num_classes
        base = r3d_18(weights=R3D_18_Weights.DEFAULT if pretrained else None)
        self.features = nn.Sequential(*list(base.children())[:-1])
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(512, 256), nn.ReLU(), nn.Dropout(dropout * 0.5),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        feat = self.features(x).flatten(1)
        return self.classifier(feat)

    def freeze_backbone(self):
        for p in self.features.parameters(): p.requires_grad = False
        print('🧊 Backbone frozen')

    def unfreeze_backbone(self):
        for p in self.features.parameters(): p.requires_grad = True
        print('🔥 Backbone unfrozen')

model = HeartClassifier3D(NUM_CLASSES, pretrained=True).to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f'Model params: {total_params:,}')

## 4. Training Loop

In [ ]:
def train_epoch(model, loader, criterion, optimizer, scaler):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        with torch.amp.autocast('cuda', enabled=device.type=='cuda'):
            outputs = model(images)
            loss = criterion(outputs, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer); scaler.update()
        total_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += images.size(0)
    return total_loss / total, 100 * correct / total

@torch.no_grad()
def validate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)
        total_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += images.size(0)
    return total_loss / total, 100 * correct / total

In [ ]:
# ══════ 2-PHASE TRAINING ══════
criterion = nn.CrossEntropyLoss(weight=class_weights)
scaler = torch.amp.GradScaler('cuda', enabled=device.type=='cuda')
best_val_acc = 0
patience_counter = 0
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

# Phase 1: Freeze backbone
print('═' * 50)
print('Phase 1: Training classifier head only')
print('═' * 50)
model.freeze_backbone()
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LR_PHASE1, weight_decay=WEIGHT_DECAY)

for epoch in range(FREEZE_EPOCHS):
    t0 = time.time()
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, scaler)
    val_loss, val_acc = validate(model, val_loader, criterion)
    dt = time.time() - t0
    history['train_loss'].append(train_loss); history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc); history['val_acc'].append(val_acc)
    marker = '⭐' if val_acc > best_val_acc else ''
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({'model_state_dict': model.state_dict(), 'epoch': epoch, 'val_accuracy': val_acc}, f'{OUTPUT_DIR}/3dcnn_best.pth')
    print(f'[{epoch+1}/{FREEZE_EPOCHS}] {dt:.0f}s | Loss: {train_loss:.4f}/{val_loss:.4f} | Acc: {train_acc:.1f}%/{val_acc:.1f}% {marker}')

# Phase 2: Unfreeze all
print('\n' + '═' * 50)
print('Phase 2: Fine-tuning all layers')
print('═' * 50)
model.unfreeze_backbone()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR_PHASE2, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=TOTAL_EPOCHS - FREEZE_EPOCHS)

for epoch in range(FREEZE_EPOCHS, TOTAL_EPOCHS):
    t0 = time.time()
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, scaler)
    val_loss, val_acc = validate(model, val_loader, criterion)
    scheduler.step()
    dt = time.time() - t0
    history['train_loss'].append(train_loss); history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc); history['val_acc'].append(val_acc)
    marker = '⭐' if val_acc > best_val_acc else ''
    if val_acc > best_val_acc:
        best_val_acc = val_acc; patience_counter = 0
        torch.save({'model_state_dict': model.state_dict(), 'epoch': epoch, 'val_accuracy': val_acc}, f'{OUTPUT_DIR}/3dcnn_best.pth')
    else:
        patience_counter += 1
    print(f'[{epoch+1}/{TOTAL_EPOCHS}] {dt:.0f}s | Loss: {train_loss:.4f}/{val_loss:.4f} | Acc: {train_acc:.1f}%/{val_acc:.1f}% {marker}')
    if patience_counter >= PATIENCE:
        print(f'Early stopping at epoch {epoch+1}'); break

print(f'\n✅ Best validation accuracy: {best_val_acc:.1f}%')

## 5. Evaluation

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Load best model
ckpt = torch.load(f'{OUTPUT_DIR}/3dcnn_best.pth', map_location=device)
model.load_state_dict(ckpt['model_state_dict'])

# Test set evaluation
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        preds = model(images).argmax(1).cpu().numpy()
        all_preds.extend(preds); all_labels.extend(labels.numpy())

print(classification_report(all_labels, all_preds, target_names=LABELS))

# Confusion matrix
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
cm = confusion_matrix(all_labels, all_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=LABELS, yticklabels=LABELS, ax=axes[0])
axes[0].set_title('Confusion Matrix'); axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')

# Training curves
axes[1].plot(history['train_acc'], label='Train Acc', color='#00d68f')
axes[1].plot(history['val_acc'], label='Val Acc', color='#ff4757')
axes[1].axvline(FREEZE_EPOCHS, color='gray', linestyle='--', alpha=0.5, label='Unfreeze')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy (%)'); axes[1].legend(); axes[1].set_title('Training Curves')
plt.tight_layout(); plt.savefig(f'{OUTPUT_DIR}/training_results.png', dpi=150); plt.show()

## 6. Download Checkpoint
Download `3dcnn_best.pth` from `/kaggle/working/` and place it at:
```
models/checkpoints/3dcnn_best.pth
```

In [ ]:
ckpt_size = os.path.getsize(f'{OUTPUT_DIR}/3dcnn_best.pth') / 1e6
print(f'Checkpoint: {OUTPUT_DIR}/3dcnn_best.pth ({ckpt_size:.1f} MB)')
print(f'Best accuracy: {best_val_acc:.1f}%')
print(f'\nTo use locally:')
print(f'  1. Download 3dcnn_best.pth')
print(f'  2. Place at: models/checkpoints/3dcnn_best.pth')
print(f'  3. Start backend: python -m src.web.app')